# 02 Baseline: TNG300-3 dark matter

The first real run with the new code: dark-matter density in, T-web classes out, with train / validation / test slabs and normalization fit on the training slab. Scores on the untouched test slab are the new baseline.

Runtime: **L4 or A100 GPU** with **High-RAM**. Expect 1 to 3 hours; early stopping usually ends it sooner. The run directory is on Drive, so the best checkpoint survives a disconnect.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!pip -q install --force-reinstall --no-deps "git+https://{token}@github.com/skumagai3/deepvoid.git"
del token
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import yaml
config = {
    'name': 'tng_dm_baseline',
    'output': '/content/drive/MyDrive/deepvoid_runs',
    'data': {'inputs': ['tng300-3/dm'], 'labels': 'tng300-3/mask',
             'split': {'fractions': [0.7, 0.15, 0.15], 'buffer': 16, 'axis': 0}},
    'model': {'depth': 4, 'base_filters': 16, 'batch_norm': True},
    'training': {'crop': 64, 'batch_size': 8, 'steps_per_epoch': 200, 'val_steps': 50,
                 'epochs': 200, 'learning_rate': 1e-3, 'class_weights': None,
                 'mixed_precision': True, 'seed': 12},
}
yaml.safe_dump(config, open('/content/baseline.yaml', 'w'))

## 1. Train

In [ ]:
# Live progress: loss, val MCC, val void F1, and a truth/prediction slice each epoch.
# Refresh the TensorBoard panel while training runs; log.csv on Drive updates too.
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/deepvoid_runs/tng_dm_baseline/tensorboard

In [ ]:
from deepvoid.train import run
out = run('/content/baseline.yaml')

## 2. Predict the whole box and score the test slab

In [ ]:
import json, keras, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from deepvoid import Catalog
from deepvoid.evaluate import confusion_matrix, scores
from deepvoid.labels import CLASS_NAMES
from deepvoid.predict import predict_volume
from deepvoid.preprocess import Normalizer
from deepvoid.split import Region

cat = Catalog()
model = keras.models.load_model(out / 'best.keras')
norm = Normalizer(**json.load(open(out / 'normalization.json'))['tng300-3/dm'])
x = norm(cat.load('tng300-3/dm').data)
y = cat.load('tng300-3/mask').data

probs = predict_volume(lambda b: model.predict_on_batch(b), x, crop=128, margin=16, batch_size=4)
pred = probs.argmax(-1).astype(np.uint8)
np.save(out / 'test_probs_float16.npy', probs.astype(np.float16))

test = Region(**json.load(open(out / 'split.json'))['test'])
cm = confusion_matrix(y, pred, mask=test.mask(y.shape[0]))
s = scores(cm)
json.dump(s, open(out / 'test_scores.json', 'w'), indent=2)
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))
print({k: round(v, 3) for k, v in s.items()})

In [ ]:
log = pd.read_csv(out / 'log.csv')
cmap = plt.matplotlib.colors.ListedColormap(['#2b6cb0', '#9ae6b4', '#f6ad55', '#c53030'])
k = (test.start + test.stop) // 2
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].plot(log['loss'], label='train'); ax[0].plot(log['val_loss'], label='val'); ax[0].legend(); ax[0].set_title('loss')
for a, (t, im) in zip(ax[1:], [('truth', y[k]), ('prediction', pred[k])]):
    a.imshow(im.T, origin='lower', cmap=cmap, vmin=0, vmax=3, interpolation='nearest'); a.set_title(f'{t}, test slab x={k}'); a.axis('off')
plt.tight_layout(); plt.show()

Paste the printed scores (and the figure, if you like) back to Claude.